In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day03" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w6" / "day03"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [2]:
import shutil                       

from sqlalchemy import func, select

from app.db.seed_data import DOCUMENTS      
from app.db.session import session_scope
from app.models.document import Chunk, DocumentVersion
from app.services import document_service

MATERIALS = next((p for p in (SAMPLES, ROOT.parent / "seed_docs") if p.exists()), None)
if MATERIALS is None:                    
    print(" 재료 폴더가 없습니다 — sandbox\\w5\\day01\\samples 도, 프로젝트 폴더 옆 seed_docs 도 없습니다.")
    print("   이미 uploads/ 에 있는 파일만 적재합니다. \n")

rows = []
for doc in DOCUMENTS:
    for v in doc["versions"]:
        dst = ROOT / v["file_path"]         
        if not dst.exists() and MATERIALS is not None:
            src = next(MATERIALS.glob(f"{doc['id']}_*_{v['version']}.{v['file_format']}"), None)
            if src is None:
                print(f"  {doc['id']} {v['version']} : 재료가 없습니다 — 이 줄만 건너뜁니다")
                continue
            dst.parent.mkdir(exist_ok=True)
            shutil.copy2(src, dst)           
        with session_scope() as s:
            dv = s.scalar(select(DocumentVersion).where(
                DocumentVersion.doc_id == doc["id"], DocumentVersion.version == v["version"]))
            real = s.scalar(select(func.count()).select_from(Chunk)
                            .where(Chunk.version_id == dv.id)) if dv else 0
        if dv is not None and real == 0 and dst.exists():
            document_service.ingest_document(doc_id=doc["id"], version=v["version"], path=str(dst))
        rows.append((doc["id"], v["version"], v["file_format"], v["status"], v["chunk_count"]))

total = table_chunks = 0
with session_scope() as s:
    for doc_id, version, fmt, status, claimed in sorted(rows):
        dv = s.scalar(select(DocumentVersion).where(
            DocumentVersion.doc_id == doc_id, DocumentVersion.version == version))
        real = s.scalar(select(func.count()).select_from(Chunk).where(Chunk.version_id == dv.id))
        total += real
        table_chunks += s.scalar(select(func.count()).select_from(Chunk)
                                 .where(Chunk.version_id == dv.id, Chunk.kind == "표"))
        print(f"{doc_id} {version:<5}{fmt:<6}{status:<5} 청크 {real:>3}   (시드가 적어 둔 값 {claimed:>3})")

print()
print(f"여덟 줄 · 청크 합계 {total}  (시드의 주장 합계 {sum(r[4] for r in rows)})  ·  표 청크 {table_chunks}")

DOC-HR-002 v3.1 pdf   현행    청크  24   (시드가 적어 둔 값 112)
DOC-HR-014 v1.1 docx  만료    청크  24   (시드가 적어 둔 값  79)
DOC-HR-014 v2.0 docx  현행    청크  53   (시드가 적어 둔 값  84)
DOC-HR-021 v1.2 docx  현행    청크  21   (시드가 적어 둔 값  41)
DOC-PM-005 v2.0 pdf   현행    청크  17   (시드가 적어 둔 값  87)
DOC-PU-007 v4.0 pdf   현행    청크  25   (시드가 적어 둔 값 156)
DOC-SE-003 v2.2 pdf   현행    청크  20   (시드가 적어 둔 값  98)
DOC-SE-011 v1.0 docx  현행    청크  21   (시드가 적어 둔 값  33)

여덟 줄 · 청크 합계 205  (시드의 주장 합계 690)  ·  표 청크 15


In [3]:
from app.db.session import session_scope
from app.rag import embedder, store 

QUESTION = "출장 갈 때 숙박비 얼마 나와요?"

query_vec = embedder.embed_query(QUESTION)
with session_scope() as s:
    hits = store.vector_search(s, query_vec, limit=12) 

print(f"질문 : {QUESTION} -> {len(hits)}건")
print("#\t\t문서\t\t\t판\t\t상태\t\t등급\t\t점수\t\t조각")
for i, h in enumerate(hits, 1):
    print(f"{i}  {h['title']}  {h['version']}  {h['version_status']}  {h['security_level']}  {h['score']}  {h['locator']}")

15:40:53 INFO     sentence_transformers.base.model: No device provided, using cpu
15:40:53 INFO     sentence_transformers.base.model: Loading SentenceTransformer model from C:\hw\hw\python_basic\models\bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

15:40:56 INFO     app.integrations.local_embed: 로컬 임베딩 모델 준비: C:\hw\hw\python_basic\models\bge-m3 / 1024차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

질문 : 출장 갈 때 숙박비 얼마 나와요? -> 12건
#		문서			판		상태		등급		점수		조각
1  국내출장 여비 규정  v2.0  현행  일반  0.6901  제3장 제2절 제14조(숙박비) · 별표1
2  국내출장 여비 규정  v1.1  만료  일반  0.6591  제3장 제9조(식비)
3  국내출장 여비 규정  v2.0  현행  일반  0.6582  제3장 제1절 제12조(식비) · 별표1
4  국내출장 여비 규정  v2.0  현행  일반  0.6555  표2
5  국내출장 여비 규정  v2.0  현행  일반  0.6492  표3
6  국내출장 여비 규정  v2.0  현행  일반  0.6465  제4장 제22조(장기 출장)
7  국내출장 여비 규정  v2.0  현행  일반  0.6445  제2장 제6조(출장의 구분)
8  국내출장 여비 규정  v2.0  현행  일반  0.6422  제3장 제2절 제16조(숙박비의 예외)
9  국내출장 여비 규정  v2.0  현행  일반  0.6335  제3장 제1절 제11조(일비) · 별표1
10  국내출장 여비 규정  v1.1  만료  일반  0.6284  제3장 제11조(숙박비) · 별표1
11  국내출장 여비 규정  v1.1  만료  일반  0.6244  제2장 제5조(출장의 구분)
12  국내출장 여비 규정  v1.1  만료  일반  0.6199  제1장 제2조(정의)


In [4]:
from datetime import timedelta
from app.models import Document
with session_scope() as s:
    for document in s.scalars(select(Document).order_by(Document.id)).all():
        currents = [v for v in document.versions if v.status == '현행']
        if len(currents) <= 1:
            print(f'{document.id} : 이미 현행이 하나입니다 — 바꿀 것이 없습니다')
            continue
        keep = max(currents, key=lambda v: v.version)
        dropped = []
        for v in currents:
            if v is keep:
                continue
            v.status = '만료'
            v.expires_at = keep.effective_from - timedelta(days=1)
            v.index_status = '보관'
            dropped.append(v.version)
        print(f'{document.id} : 현행 = {keep.version} · 만료로 바꾼 판 {len(dropped)}개')


DOC-FI-009 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-HR-002 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-HR-014 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-HR-021 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-PM-005 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-PU-007 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-SE-003 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-SE-011 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다


In [5]:
import importlib

from app.rag import policy
importlib.reload(policy)

DFT = policy.DEFAULT

print("만료 버전 : ", policy.version_allowed("만료", policy=DFT, question="숙박비 얼마?"))
print("만료 버전 : ", policy.version_allowed("만료", policy=DFT, question="예전 숙박비 얼마?"))
print("만료 버전 : ", policy.clearance_allowed("일반", "대외비", policy=DFT))
print("만료 버전 : ", policy.clearance_allowed("대외비", "대외비", policy=DFT))
print("만료 버전 : ", policy.version_allowed("만료", policy=policy.SearchPolicy(current_only=False), question="숙박비 얼마?"))

만료 버전 :  False
만료 버전 :  True
만료 버전 :  False
만료 버전 :  True
만료 버전 :  True


In [6]:
importlib.reload(policy)
USER = {'id': '2019-0412', 'name': '김민준', 'clearance': '일반', 'dept_id': 'INFRA2'}
rows = [{'title': '국내출장 여비 규정', 'version': 'v2.0', 'version_status': '현행', 'security_level': '일반'}, {'title': '국내출장 여비 규정', 'version': 'v1.1', 'version_status': '만료', 'security_level': '일반'}, {'title': '구매·계약 규정', 'version': 'v4.0', 'version_status': '현행', 'security_level': '대외비'}, {'title': '정보보안 지침', 'version': 'v2.2', 'version_status': '만료', 'security_level': '대외비'}]
kept, reasons = policy.filter_candidates(rows, user=USER, question='출장 갈 때 숙박비 얼마 나오나요?')
print(f'통과한 후보 ({len(kept)})')
for r in kept:
    print(' ', r['title'], r['version'])
print()
print(f'걸러진 이유 ({len(reasons)})')
for why in reasons:
    print(' ', why)


통과한 후보 (1)
  국내출장 여비 규정 v2.0

걸러진 이유 (3)
  국내출장 여비 규정 v1.1 - 만료 버전
  구매·계약 규정 - 보안등급(대외비) 차단
  정보보안 지침 v2.2 - 만료 버전


In [7]:
import app.core.config as config_mod
from app.rag import embedder, store
importlib.reload(config_mod)
settings = config_mod.get_settings()
print(f'임계값 {settings.retrieval_threshold} · top_k {settings.retrieval_top_k}')

"""질문 하나를 검색해 정책으로 거르고, 최고 점수가 임계값 위인지 아래인지 찍는다."""
def ask(question: str) -> None:
    vec = embedder.embed_query(question)
    with session_scope() as s:
        found = store.vector_search(s, vec, limit=settings.retrieval_top_k)
    kept, _ = policy.filter_candidates(found, user=USER, question=question)
    best = max((r['score'] for r in kept), default=0.0)
    print()
    print('질문 :', question)
    if best < settings.retrieval_threshold:
        print(f'  최고 점수 {best:.2f} / 임계값 {settings.retrieval_threshold}  →  근거 부족')
    else:
        print(f'  최고 점수 {best:.2f} / 임계값 {settings.retrieval_threshold}  →  상위 세 줄')
        for r in kept[:3]:
            print(f'    {r['title']} {r['version']} {r['locator']}')
ask('출장 갈 때 숙박비 얼마 나오나요?')
ask('반려동물 동반 출장이 되나요?')


임계값 0.55 · top_k 12


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


질문 : 출장 갈 때 숙박비 얼마 나오나요?
  최고 점수 0.69 / 임계값 0.55  →  상위 세 줄
    국내출장 여비 규정 v2.0 제3장 제2절 제14조(숙박비) · 별표1
    국내출장 여비 규정 v2.0 제3장 제1절 제12조(식비) · 별표1
    국내출장 여비 규정 v2.0 표2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


질문 : 반려동물 동반 출장이 되나요?
  최고 점수 0.48 / 임계값 0.55  →  근거 부족
